# Agent框架与策略分析

## 其他Agent认知框架

### Plan-and-Execute

计划与执行（Plan-and-Execute）框架侧重于先规划一系列的行动，然后执行。这个框架可以使大模型能够先综合考虑任务的多个方面，然后按照计划进行行动。应用在比较复杂的项目管理中或者需要多步决策的场景下会比较合适。

<img src="../images/plan-and-execute.png" width=70% style="display: block; margin: auto;">

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_experimental.plan_and_execute import PlanAndExecute, load_agent_executor, load_chat_planner
from langchain import SerpAPIWrapper
from langchain.agents.tools import Tool
from langchain import LLMMathChain
from dotenv import load_dotenv
load_dotenv()
# 配置API密钥和基础URL
llm = ChatOpenAI()

# https://serpapi.com/dashboard
# 创建工具
search = SerpAPIWrapper() 
llm_math_chain = LLMMathChain(llm=llm, verbose=True)

# 定义工具列表
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="用于回答关于当前事件的问题"
    ),
    Tool(
        name="Calculator",
        func=llm_math_chain.run,
        description="用于计算或解决问题"
    )
]

# 加载规划器和执行器
planner = load_chat_planner(llm)
executor = load_agent_executor(llm, tools, verbose=True)

# 创建Plan and Execute代理
agent = PlanAndExecute(planner=planner, executor=executor, verbose=True)

# 运行代理解决实际问题
print(agent.run("在纽约，100美元能买几束玫瑰？"))


: 

### Self-Ask

自问自答（Self-Ask）框架这个允许大模型对自己提出问题并回答，来增强对问题的理解以提高回答质量，这个框架在需要深入分析或者提供创造性解决方案下可以比较适合，例如创意写作。

<img src="../images/Self-Ask.webp" width=70% style="display: block; margin: auto;">

In [ ]:

import os
from langchain import hub
from langchain.agents import AgentExecutor, create_self_ask_with_search_agent
from langchain_community.tools.tavily_search import TavilyAnswer
from dotenv import load_dotenv
load_dotenv()
# 配置模型
from langchain_fireworks import Fireworks

llm = Fireworks(
    # https://fireworks.ai/account/billing
    api_key=os.getenv("FIREWORKS_API_KEY"),
    # model="accounts/fireworks/models/mixtral-8x7b-instruct",
    model="accounts/fireworks/models/llama-v3p1-405b-instruct",
    max_tokens=256)

tools = [TavilyAnswer(max_results=1, name="Intermediate Answer",tavily_api_key=os.getenv("TAVILY_API_KEY"))]
#获取prompt
prompt = hub.pull("hwchase17/self-ask-with-search")
print(prompt)
agent = create_self_ask_with_search_agent(llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True,handle_parsing_errors=True)
print(agent_executor.invoke({"input": "上一届的美国总统是谁?"}))

### Thinking and Self-Refection

思考并自我反思（Thinking and Self-Refection）框架主要用于模拟和实现复杂决策过程，通过不断自我评估和调整，使系统能够学习并改进决策过程，从而在面对复杂问题是作出更加有效的决策。

**langgraph**  
**rankrag**

<img src="../images/Thinking and Self-Reflection.webp" width=70% style="display: block; margin: auto;">

## ReAct框架回顾

<img src="../images/react回顾.webp" width=40% style="display: block; margin: auto;">

在langchain使用Agent中，我们重点需要理解下面4个元素。
+ 1、llm: 提供逻辑的引擎，负责生成预测和处理输入。
+ 2、prompt: 负责指导模型，形成推理框架
+ 3、tools: 外部工具的使用 包含数据增强、清洗、搜索引擎、api等等。
+ 4、Agent Executor: 负责调用合适的外部工具，并管理整个流程。


In [ ]:
# 导入环境变量
from dotenv import load_dotenv
load_dotenv()

# 初始化大模型
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4-turbo-preview',
             temperature=0.5)

# 设置工具
from langchain_community.agent_toolkits.load_tools import load_tools
tools = load_tools(["serpapi", "llm-math"], llm=llm)

# 设置提示模板
from langchain.prompts import PromptTemplate
template = ('''
    '尽你所能用中文回答以下问题。如果能力不够你可以使用以下工具:\n\n'
    '{tools}\n\n
    Use the following format:\n\n'
    'Question: the input question you must answer\n'
    'Thought: you should always think about what to do\n'
    'Action: the action to take, should be one of [{tool_names}]\n'
    'Action Input: the input to the action\n'
    'Observation: the result of the action\n'
    '... (this Thought/Action/Action Input/Observation can repeat N times)\n'
    'Thought: I now know the final answer\n'
    'Final Answer: the final answer to the original input question\n\n'
    'Begin!\n\n'
    'Question: {input}\n'
    'Thought:{agent_scratchpad}'
    '''
)
prompt = PromptTemplate.from_template(template)

# 初始化Agent
from langchain.agents import create_react_agent
agent = create_react_agent(llm, tools, prompt)

# 构建AgentExecutor
from langchain.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent,
                               tools=tools,
                               handle_parsing_errors=True,
                               verbose=True)

# 执行AgentExecutor
agent_executor.invoke({"input": """目前市场上玫瑰花的一般进货价格是多少？\n如果我在此基础上加价5%，应该如何定价？"""})




## AgentExecutor运行机制

##### 具体上课进行源码分析

In [ ]:
base.py agent.py 
on_chain_start
self._call
_take_next_step

## 通过llamaindex实现ReAct RAG Agent

In [ ]:
# 加载电商财报数据
from llama_index.core import SimpleDirectoryReader
from dotenv import load_dotenv
load_dotenv()
A_docs = SimpleDirectoryReader(
    input_files=["./A.pdf"]
).load_data()
B_docs = SimpleDirectoryReader(
    input_files=["./B.pdf"]
).load_data()



# 从文档中创建索引
from llama_index.core import VectorStoreIndex
A_index = VectorStoreIndex.from_documents(A_docs)
B_index = VectorStoreIndex.from_documents(B_docs)

# 持久化索引（保存到本地）
from llama_index.core import StorageContext
A_index.storage_context.persist(persist_dir="./storage/A")
B_index.storage_context.persist(persist_dir="./storage/B")


# 从本地读取索引
from llama_index.core import load_index_from_storage
try:
    storage_context = StorageContext.from_defaults(
        persist_dir="./storage/A"
    )
    A_index = load_index_from_storage(storage_context)

    storage_context = StorageContext.from_defaults(
        persist_dir="./storage/B"
    )
    B_index = load_index_from_storage(storage_context)

    index_loaded = True
except:
    index_loaded = False


# 创建查询引擎
A_engine = A_index.as_query_engine(similarity_top_k=3)
B_engine = B_index.as_query_engine(similarity_top_k=3)


# 配置查询工具
from llama_index.core.tools import QueryEngineTool
from llama_index.core.tools import ToolMetadata
query_engine_tools = [
    QueryEngineTool(
        query_engine=A_engine,
        metadata=ToolMetadata(
            name="A_Finance",
            description=(
                "用于提供A公司的财务信息 "
            ),
        ),
    ),
    QueryEngineTool(
        query_engine=B_engine,
        metadata=ToolMetadata(
            name="B_Finance",
            description=(
                "用于提供A公司的财务信息 "
            ),
        ),
    ),
]


# 配置大模型
from llama_index.llms.openai import OpenAI
llm = OpenAI(model="gpt-4")


# 创建ReAct Agent
from llama_index.core.agent import ReActAgent
agent = ReActAgent.from_tools(query_engine_tools, llm=llm, verbose=True)


# 让Agent完成任务
print(agent.chat("Compare the sales of the two companies"))
